# Домашнее задание по нейросетевым конвейерам обработки изображений

В данном задании вам предстоит собрать пайплайн обработки изображений изображений. По результатам каждого модуля, вам предстоит собрать один целостный пайплайн, состоящий из "лучших" (полученных на предложенном датасете) блоков.  

## В нем вы:
- Обучите с нуля модель для задачи дебайеринга 
- Дообучите FC4 модель для задачи баланса белого
- Обучите end-to-end пайплайн RAW->sRGB
- Проведете сравнительный анализ нейросетевых блоков с их классическими вариантами и соберете из них полный пайплайн 


***Будем симулировать пайплайн для камеры Canon 600D***

Основную часть методов надо будет имплементировать в *.py файлах, в этом .ipynb файле предстоит провести анализ ошибок, а также проверить, что ваши реализации функций визуально-адекватные

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Дебайеринг

Матрица сенсора камеры записывает одно число на каждый свой пиксель. Чтобы из этого двумерного массива получить нам знакомый трехмерный массив РГБ нужно сделать дебайеринг / демозайкинг. 
Для тестирования методов дебайеринга нужно иметь пару изображений до дебайеринга и его РГБ вариант.
Один из способов получить такой набор изображений — использовать гиперспектральные изображения (ГСИ)

В данной работе вам дается гиперспектральный датасет из ~400 фотографий. Для каждой фотографии есть бинарная маска с цветовой мишени и белого патча.

Задача:
- С помощью ГСИ насимулировать изображения до и после дебайеринга для камеры Canon600D. Фиксируем RGGB паттерн
- Продумать аугментации которые можно использовать для этой задачи
- Имплементировать и обучить сверточнуть модель
- Сравнить качество обученной модели и Menon [PSNR, SSIM]
- Посмотреть на активации слоев модели 

Ответить на вопросы:
- В чем заключается алгоритм Menon? Какие преимущества имеют сверточные неройсети по сравнению с Menon?
- ГСИ не является единственным способом создания граундтруза. Как вы бы решали эту задачу без ГСИ?

In [ ]:
# Подсказка как симулировать изображения до и после дебайеринга. 
# Каждый пиксель до дебайеринга получен после прохождения красного, синего, зеленого светофильтра, в фиксированном порядке.
# Вспомните формулу линейной модели формирования изображения.

In [ ]:
# При обучении нейронной сети важно будет использовать аугментации. 
# Подумайте, какие аугментации для этой задачи будут полезны. 
# Если вы придумали какие-то интересные аугментации, но не смогли их реализовать, опишите их в отчете.

# Разделите выборку на трейн и тест 80%/20%. Обучите модель, выберите сами лосс который считаете подходящим.  
# Вставьте график сходимости обучения модели. Посчитайте метрики на тесте с помощью вашей модели и Menon.

from debayering.metrics import MSE, PSNR, SSIM

# 2. AWB & CST

В данной части предлагается дообучить модель из статьи "FC4 : Fully Convolutional Color Constancy with Confidence-weighted Pooling" для задачи баланса белого и RPCC для преобразования цветовых координат.

В отличие от основной статьи мы используем здесь SqueezeNet. Посмотрите в статью чтобы правильно прокинуть лосс. Можно также ознакомиться с  https://github.com/matteo-rizzo/fc4-pytorch/tree/main для вдохновления.

Задача:
- Дообучить FC4 модель на предложенном датасете для задачи баланса белого
- Провести аугментацию данных, в том числе путем симуляции новой точки белого [data/illuminants.npy]
- Сравнить обученный метод с классическим grayness index по метрикам angle
- Посмотреть confidence map 
- Сделать преобразование цветовых координат через Root-Polynomial

Ответить на вопросы:
- Почему была предложено дообучить FC4 модель? В чем идея метода?
- Как работает grayness index? 
- Предсказывает ли FC4 одну точку белого или несколько?

In [ ]:
# обучите FC4 на предложенном датасете, обязательно используйте аугментации новыми точками белого. 
# замаскируйте/обрежьте картинки, чтобы там не было белого патча. 
# после обучения, попробуйте на полной картинке и посмотрите на confidence map. Будет ли сеть смотреть на белый патч?

# 3. end-to-end ISP

В этом задании предстоит собрать полный пайплайн. Состоит он из следующих шагов:
RAW изображение -> дебайеринг -> AWB -> CST -> sRGB

Надо рассмотреть следующие варианты: 
- а) все блоки классические (menon, grayness index, rpcc)
- б) выбраны лучшие блоки (гибрид из нейросетевых блоков и классических, с лучшими метриками по каждой задаче)

Также надо будет обучить end-to-end неросетевой пайплайн. Такой подход не делит конвейер на блоки, тем самым ошибка не накапливается.
В качестве архитектуры будем использовать MobileNet V3 -- https://docs.pytorch.org/vision/main/models/generated/torchvision.models.mobilenet_v3_small.html#torchvision.models.mobilenet_v3_small

Считайте что ваш граундтруз -- гиперспектральное изображение переведенное сразу в sRGB.

Задача:
- Собрать пайплайны а) и б) и посчитать метрику deltaE
- Обучить MobileNetV3-small  для end-to-end ISP
- Сравнить качество полностью классического, полностью нейросетевого и гибридного решения.

In [ ]:
from _deltaE import DeltaE
